# Compare complete-profile depth selections
Run 00–01 once with `full_500m`, then again with `shallow_200m`. This notebook compares **shared exact depths** and selection overlap. It does not subtract 181 m displacement from 515 m displacement as a selection test.

Intervals are the separate runs' pointwise intervals, not confidence intervals for their difference. Changes can reflect population composition; these are descriptive sensitivity comparisons. Inspect controls printed below—other settings should match for a depth-only interpretation.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HERE = Path.cwd().resolve()
ANALYSIS = next((p for p in (HERE, *HERE.parents) if (p/'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subdirectory')
WORK = ANALYSIS/'esp_population_composites'
for p in (ANALYSIS, WORK):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import seacofs_tilt_tools as tilt
import population_tools as pop
pd.set_option('display.max_columns', 60)


In [2]:
OUTPUT_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/esp_population_composites')
RUNS = {}  # optionally {'full_500m': Path(...), 'shallow_200m': Path(...)}
if not RUNS:
    # Latest completed run for each declared preset; never mix in incomplete audits.
    for preset in ('full_500m', 'shallow_200m'):
        candidates=[]
        for path in OUTPUT_ROOT.glob('*/provenance.json'):
            saved=json.loads(path.read_text())
            if saved['config'].get('depth_preset') != preset: continue
            try:
                pop.load_saved_results(path.parent)
            except (FileNotFoundError, ValueError):
                continue
            candidates.append(path)
        if not candidates:
            raise FileNotFoundError(f'No completed {preset} run; execute 00 then 01 with that preset')
        RUNS[preset] = max(candidates, key=lambda p:p.stat().st_mtime).parent
for label, path in RUNS.items():
    print(label, path)
    print(json.dumps(json.loads((path/'provenance.json').read_text())['config'], indent=2))
comparison, overlap = pop.compare_saved_runs(RUNS)
if comparison.empty:
    raise ValueError('These runs have no shared exact depths')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(comparison.round(6))
    display(overlap)
# Name outputs by source run IDs to keep comparisons separate.
import hashlib
key=hashlib.sha256(json.dumps({k:str(v) for k,v in RUNS.items()},sort_keys=True).encode()).hexdigest()[:12]
report=OUTPUT_ROOT/'comparisons'/key
report.mkdir(parents=True, exist_ok=True)
comparison.to_csv(report/'shared_depth_intervals.csv', index=False)
overlap.to_csv(report/'membership_overlap.csv', index=False)
(report/'sources.json').write_text(json.dumps({k:str(v) for k,v in RUNS.items()},indent=2))


FileNotFoundError: No completed shallow_200m run; execute 00 then 01 with that preset

In [ ]:
groups=list(comparison.group.unique())
fig,axes=plt.subplots(len(groups),2,figsize=(11,3*len(groups)),squeeze=False,constrained_layout=True)
for i,group in enumerate(groups):
    part=comparison.loc[comparison.group.eq(group)]
    for j,component in enumerate(part.component.unique()):
        ax=axes[i,j]
        for label, selected in part.loc[part.component.eq(component)].groupby('run',sort=False):
            selected=selected.sort_values('depth_m')
            ax.plot(selected['mean'],selected.depth_m,'o-',label=label)
            ax.fill_betweenx(selected.depth_m,selected.ci_low,selected.ci_high,alpha=.2)
        ax.axvline(0,color='.4',ls=':')
        ax.invert_yaxis()
        ax.set(title=f'{group}: {component}',xlabel=f'Centre displacement ({part.units.iloc[0]})',ylabel='Depth (m)')
        ax.legend()
fig.suptitle('Depth-selection sensitivity at shared exact levels; separate pointwise 95% intervals')
fig.savefig(report/'shared_depth_comparison.png',dpi=180)
plt.show()
print('Comparison saved:',report)
